# 05 - Batch and Online Evaluation

This notebook moves from individual evaluation runs to repeatable checks over past and live traffic.

You will:

1. Choose between a planned dataset, a historical batch job, and live monitoring.
2. Turn an evaluation score into a CI pass/fail check.
3. Evaluate a sample of new requests automatically.
4. Find evaluation logs and metrics in CloudWatch.

**Estimated time:** 35-50 minutes  
**Creates AWS resources:** Optional batch jobs and online evaluation configuration.  
**Feature status:** Batch evaluation is public preview as of August 14, 2026. Batch evaluation API activity is not currently recorded as CloudTrail events.

## 1. Choose the right execution mode

| Need | Mechanism |
|---|---|
| Understand one recent conversation | `agentcore run eval --session-id ...` |
| Send a planned set of scenarios to the agent | Dataset runner or managed dataset |
| Evaluate many conversations that already happened | Batch evaluation |
| Automatically evaluate a percentage of new conversations | Online evaluation |

Online evaluation runs after the agent responds. The user does not wait for the judge. AgentCore records the trace, selects it according to the sampling rate, and evaluates it in the background.

## 2. Batch evaluation over historical sessions

Batch evaluation reads sessions that are already recorded in CloudWatch and evaluates many of them as one job. Use it when you want to score older sessions, compare results across time periods, or review a larger set before a release.

In [ ]:
import pandas as pd
from IPython.display import display

from src.workshop_utils import MODULE_ROOT, run_cli, run_cli_json

RUN_BATCH_EVALUATION = False

if RUN_BATCH_EVALUATION:
    batch_result = run_cli_json(
        "run",
        "batch-evaluation",
        "--runtime",
        "CityAnalyst",
        "--evaluator",
        "Builtin.Helpfulness",
        "Builtin.GoalSuccessRate",
        "--lookback-days",
        "1",
        "--name",
        "CityAnalystRegression",
        "--wait",
    )
    batch_summary = pd.DataFrame(
        [
            {
                "Job ID": batch_result.get("id"),
                "Name": batch_result.get("name", "CityAnalystRegression"),
                "Status": batch_result.get("status", "Submitted"),
                "Sessions": batch_result.get("sessionCount"),
                "Created": batch_result.get("createdAt"),
            }
        ]
    )
    display(batch_summary)
    print("The complete service response is available in batch_result.")
else:
    print("Set RUN_BATCH_EVALUATION=True after generating several sessions.")

A batch job has an overall status plus results for the sessions it evaluated. When you compare jobs over time, keep enough context to explain any score change:

- the evaluator version or ARN
- the runtime version or endpoint
- where the sessions came from and the time range used
- the spread of scores, not only the average
- how many sessions completed, failed, or were skipped

## 3. A CI quality gate

A quality gate turns an evaluation result into an automated release decision. It should stop the release when:

- the evaluation command fails
- no sessions are found
- evaluator results contain errors
- the selected score is below its tested threshold

The threshold below is only an example. Choose a real threshold after comparing evaluator decisions with human-labeled examples, as shown in Notebook 04.

In [ ]:
import json


def run_quality_gate(
    runtime_name: str,
    evaluator_id: str,
    threshold: float,
    lookback_days: int = 1,
) -> dict:
    if not 0 <= threshold <= 1:
        raise ValueError("threshold must be between 0 and 1.")

    payload = run_cli_json(
        "run",
        "eval",
        "--runtime",
        runtime_name,
        "--evaluator",
        evaluator_id,
        "--days",
        str(lookback_days),
    )
    run = payload.get("run", payload)
    result_rows = run.get("results", [])
    if not result_rows:
        raise RuntimeError("Quality gate found no evaluation results.")

    matching = [
        item
        for item in result_rows
        if item.get("evaluator") == evaluator_id
    ]
    if not matching:
        raise RuntimeError(f"No result found for {evaluator_id}.")

    session_scores = [
        session
        for item in matching
        for session in item.get("sessionScores", [])
    ]
    if not session_scores:
        raise RuntimeError("Quality gate found no session-level evaluation results.")

    evaluation_errors = [
        session
        for session in session_scores
        if session.get("errorCode") or session.get("errorMessage")
    ]
    if evaluation_errors:
        details = "; ".join(
            f"{item.get('sessionId', 'unknown session')}: "
            f"{item.get('errorMessage') or item.get('errorCode')}"
            for item in evaluation_errors[:5]
        )
        raise RuntimeError(
            f"Quality gate found {len(evaluation_errors)} evaluation error(s): {details}"
        )

    score = matching[0].get("aggregateScore")
    if score is None:
        raise RuntimeError("Evaluation result did not include aggregateScore.")
    if score < threshold:
        raise RuntimeError(
            f"Quality gate failed: {score:.3f} < {threshold:.3f}"
        )
    return {
        "Evaluator": evaluator_id,
        "Sessions checked": len(session_scores),
        "Score": score,
        "Required score": threshold,
        "Decision": "PASS",
    }


CALIBRATED_THRESHOLD = 0.70
RUN_QUALITY_GATE = False

if RUN_QUALITY_GATE:
    gate_result = run_quality_gate(
        runtime_name="CityAnalyst",
        evaluator_id="Builtin.Helpfulness",
        threshold=CALIBRATED_THRESHOLD,
    )
    display(pd.DataFrame([gate_result]))
else:
    print("Calibrate the threshold, then set RUN_QUALITY_GATE=True.")

In CI, first run a small set of representative agent requests. Wait for their trace data to appear, but stop waiting after a defined timeout. Then run the quality gate against those sessions. This prevents an endless wait and avoids evaluating before the evidence is ready.

## 4. Configure online evaluation

The sampling percentage controls how much new traffic is evaluated. For this workshop, `100` evaluates every request so results appear quickly. For a production application, a smaller starting point such as `1-5` percent often provides useful coverage at lower cost. Increase it when the application has higher risk, stronger review requirements, or very low traffic.

In [ ]:
import json

ONLINE_CONFIG_NAME = "CityQualityMonitor"
CREATE_ONLINE_CONFIG = False
WORKSHOP_SAMPLING_PERCENTAGE = 100

project_config = json.loads(
    (MODULE_ROOT / "agentcore" / "agentcore.json").read_text()
)
existing_online = {
    item["name"]
    for item in project_config.get("onlineEvalConfigs", [])
}

if CREATE_ONLINE_CONFIG and ONLINE_CONFIG_NAME not in existing_online:
    result = run_cli(
        "add",
        "online-eval",
        "--name",
        ONLINE_CONFIG_NAME,
        "--runtime",
        "CityAnalyst",
        "--evaluator",
        "Builtin.Helpfulness",
        "Builtin.GoalSuccessRate",
        "Builtin.ToolSelectionAccuracy",
        "--sampling-rate",
        str(WORKSHOP_SAMPLING_PERCENTAGE),
        "--enable-on-create",
    )
    print(result.stdout.strip())
    print("Online config added locally. Review agentcore.json, then deploy.")
elif CREATE_ONLINE_CONFIG:
    print(f"{ONLINE_CONFIG_NAME} is already present in agentcore.json.")
else:
    print("Set CREATE_ONLINE_CONFIG=True to add the workshop monitor.")

Review the proposed infrastructure changes before deploying. The flag below keeps the AWS change explicit and deploys to the workshop's `default` target without opening an interactive prompt.

AgentCore configuration names use letters, numbers, and underscores, so this example uses `CityQualityMonitor`.

In [ ]:
DEPLOY_ONLINE_CONFIG = False

if DEPLOY_ONLINE_CONFIG:
    diff = run_cli("deploy", "--target", "default", "--diff")
    print(diff.stdout.strip())
    online_deploy_result = run_cli_json(
        "deploy", "--target", "default", "--yes"
    )
    print(f"Deployed {ONLINE_CONFIG_NAME} to the default target.")
else:
    print("Set DEPLOY_ONLINE_CONFIG=True after reviewing agentcore.json.")

## 5. Generate controlled traffic

Run this only after the online configuration is deployed and active. The prompts cover successful lookups, comparisons, calculations, missing data, and a greeting so the monitor sees several kinds of behavior.

In [ ]:
from src.workshop_utils import RuntimeInvoker, load_runtime_info, make_session_id

GENERATE_TRAFFIC = False
prompts = [
    "What are the workshop facts for Seattle, WA?",
    "Compare Boston, MA with Miami, FL.",
    "Calculate density for 400000 people and 80 square miles.",
    "What is the population of Atlantis, CA?",
    "Hello. What can you help me with?",
]

if GENERATE_TRAFFIC:
    runtime = load_runtime_info()
    runtime_invoker = RuntimeInvoker(runtime)
    traffic_rows = []
    traffic_results = []
    for prompt in prompts:
        session_id = make_session_id("online")
        response = runtime_invoker.invoke(
            prompt=prompt,
            session_id=session_id,
        )
        traffic_results.append(response)
        answer = response.get("response", response) if isinstance(response, dict) else response
        if isinstance(answer, (dict, list)):
            answer = json.dumps(answer, indent=2)
        traffic_rows.append(
            {
                "Prompt": prompt,
                "Answer": answer,
                "Session ID": session_id,
            }
        )
    display(pd.DataFrame(traffic_rows))
else:
    print("Deploy the online config, then set GENERATE_TRAFFIC=True.")

## 6. Observe and control the monitor

Online evaluation results are written to CloudWatch after sampled traces are processed. The next cell can read recent results or pause and resume the monitor without deleting its configuration.

Pause the monitor when you need evaluation to stop temporarily, for example while investigating an incident, updating an evaluator, controlling unexpected cost, or protecting traffic that should not be evaluated.

In [ ]:
READ_ONLINE_EVAL_LOGS = False
ONLINE_MONITOR_ACTION = "none"  # one of: none, pause, resume

if READ_ONLINE_EVAL_LOGS:
    logs = run_cli(
        "logs",
        "evals",
        "--runtime",
        "CityAnalyst",
        "--since",
        "1h",
        "--limit",
        "100",
    )
    print(logs.stdout.strip() or "No online evaluation logs found.")

if ONLINE_MONITOR_ACTION not in {"none", "pause", "resume"}:
    raise ValueError("ONLINE_MONITOR_ACTION must be none, pause, or resume.")
if ONLINE_MONITOR_ACTION != "none":
    monitor_result = run_cli_json(
        ONLINE_MONITOR_ACTION, "online-eval", ONLINE_CONFIG_NAME
    )
    print(
        f"{ONLINE_CONFIG_NAME} action completed: "
        f"{ONLINE_MONITOR_ACTION}."
    )

## 7. Discover metrics before creating alarms

CloudWatch organizes metrics with three pieces of information:

- A **namespace** groups metrics from the same service or feature.
- A **metric name** identifies the value being measured.
- **Dimensions** are labels that identify the resource or configuration behind that value.

These names are case-sensitive. Discover the names produced by your deployed configuration before creating an alarm.

In [ ]:
import boto3
from botocore.config import Config
from src.workshop_utils import load_runtime_info

runtime = load_runtime_info()
cloudwatch = boto3.client(
    "cloudwatch",
    region_name=runtime.region,
    config=Config(
        retries={"total_max_attempts": 5, "mode": "adaptive"},
        connect_timeout=5,
        read_timeout=30,
    ),
)

EVALUATION_METRICS_NAMESPACE = "Bedrock-AgentCore/Evaluations"
paginator = cloudwatch.get_paginator("list_metrics")
discovered = {}
for page in paginator.paginate(Namespace=EVALUATION_METRICS_NAMESPACE):
    for metric in page.get("Metrics", []):
        dimension_names = tuple(
            sorted(item["Name"] for item in metric.get("Dimensions", []))
        )
        discovered[(metric["MetricName"], dimension_names)] = {
            "namespace": EVALUATION_METRICS_NAMESPACE,
            "metric_name": metric["MetricName"],
            "dimensions": list(dimension_names),
        }

discovered_metrics = sorted(
    discovered.values(),
    key=lambda item: (item["metric_name"], item["dimensions"]),
)
if not discovered_metrics:
    print(
        "No evaluation metrics were found yet. Generate evaluated traffic, "
        "wait for CloudWatch ingestion, and run this cell again."
    )
if discovered_metrics:
    metrics_df = pd.DataFrame(
        [
            {
                "Namespace": item["namespace"],
                "Metric": item["metric_name"],
                "Dimension names": ", ".join(item["dimensions"]) or "None",
            }
            for item in discovered_metrics
        ]
    )
    display(metrics_df)

Create alarms only for metrics whose meaning and threshold you have tested. Include enough identifiers to find the related sessions and traces, but do not send raw prompts or other sensitive content to an SNS topic with broad access.

## 8. Before using this in production

- Choose a sampling rate that balances risk, traffic volume, and evaluation cost.
- Set retention and encryption for CloudWatch logs and notification topics.
- Limit who can read traces and evaluator explanations.
- Remove or protect personal and sensitive data before exporting trace data.
- Confirm whether the judge model processes data in another Region.
- Alert on evaluator errors separately from low agent scores.
- Version evaluator definitions and thresholds with the application.
- Continue reviewing a small human-labeled sample so you can detect changes in judge behavior.

## 9. Checkpoint

You can now connect:

- a planned dataset before release
- evaluation of historical sessions
- an automated CI decision
- evaluation of a sample of live traffic

Continue to [06 - Simulation and Optimization](06-simulation-and-optimization.ipynb) to generate harder conversations, find failure patterns, and test improvements.